# Entraînement PPO : rebonds successifs

Choisir le kernel **so101-robotic-learning**, puis exécuter les cellules dans l'ordre.
EGL n'est pas imposé : sur un serveur Linux sans écran, configurer le backend avant tout import MuJoCo si nécessaire.

La récompense vaut **5 × rebonds + 0.05 × shaping − 5 en cas d'échec − pénalité d'énergie − pénalité de distance au segment** (chute, raquette au sol ou balle à plus de 2 m de la raquette). La réussite demande
5 rebonds consécutifs par défaut. Les checkpoints sont sélectionnés selon les métriques
physiques ; leurs scores ne sont pas directement comparables aux anciens entraînements.
Les logs sont écrits dans `runs/ppo_bounce/`. Modèles, rapports et vidéos utilisent des sous-dossiers datés distincts.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
from IPython.display import JSON, Markdown, display

# Works from the repository root or any of its subdirectories.
root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file()
             and (path / "src/so101_robotic_learning").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt ou l'un de ses sous-dossiers.")
xml_path = root / "assets/mjcf/so101_new_calib copy.xml"
if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

# MuJoCo selects the platform backend. Do not force EGL on Windows.
from so101_robotic_learning.bounce_rl.env.bounce_events import BounceConfig

from so101_robotic_learning.bounce_rl.rl import train_ppo


## Paramètres

`reward`, si fourni à `train()`, désigne le shaping auxiliaire. Les paramètres d'environnement
passent par `env_kwargs`. Pour utiliser un historique, modifier `latent_steps` **avant** de
lancer l'entraînement et conserver exactement la même configuration lors du test du modèle.
PPO et SAC utilisent ici la même tâche et le même seuil de réussite.


In [ ]:
run_id = "ppo_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_root = root

env_kwargs = {
    "camera_name": "gripper_cam",  # policy input only
    "render_profile": "train_fast",  # same profile for training and evaluation
    "video_render_profile": "standard",  # external camera only
    "video_camera_name": "overview_cam",  # visualization only
    "video_width": 320,
    "video_height": 240,
    "bounce_config": BounceConfig(success_bounces=5, max_ball_paddle_distance=2.0),
    "bounce_reward_weight": 5.0,
    "shaping_reward_weight": 0.05,
    "fall_penalty": 5.0,
    "energy_penalty_weight": 0.01,  # absolute mechanical work (J), including braking
    "ball_segment_penalty_weight": 1.0,  # time-integrated distance to the vertical segment
    "ball_segment_start_height": 0.2,  # lower endpoint above paddle_face (m)
    "ball_segment_length": 1.0,  # segment length along world Z (m)
    "max_episode_steps": 1024,
    "frame_skip": 20,
    "latent_steps": np.array([], dtype=np.int32),
}
train_config = {
    "xml_path": xml_path,
    "root": run_root,
    "model_dir_name": f"models/{run_id}",
    "env_kwargs": env_kwargs,
    "total_timesteps": 20_000,
    "nb_vec_env_train": 1,
    "nb_vec_env_test": 1,
    "learning_rate": 3e-4,
    "n_steps": 256,
    "batch_size": 64,
    "n_epochs": 10,
    "eval_every_episodes": 10,
    "n_eval_episodes": 3,
    "eval_max_steps": env_kwargs["max_episode_steps"],
    "train_render_mode": None,
    "eval_render_mode": None,  # recording is enabled independently of the viewer
    "save_eval_video": True,
    "tensorboard_log": True,
    "run_subdir": "ppo_bounce",
    "device": "auto",
}
RUN_TENSORBOARD = True  # optional inline dashboard
RUN_POLICY_TEST = False  # optional evaluation of the best checkpoint
print(f"Logs TensorBoard : {run_root / 'runs/ppo_bounce'}")
print(f"Modèles et rapports : {run_root / train_config['model_dir_name']}")


## Train


In [ ]:
train_ppo.train(**train_config)
